<a href="https://colab.research.google.com/github/Swayam-dot-cmd/neural-information-retrieval/blob/main/notebooks/02_dense_bge.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# Install everything (run this FIRST every time)
!pip install -q beir sentence-transformers scikit-learn

In [ ]:
from beir.datasets.data_loader import GenericDataLoader
from beir import util

from sentence_transformers import SentenceTransformer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
dataset = "scifact"
url = f"https://public.ukp.informatik.tu-darmstadt.de/thakur/BEIR/datasets/{dataset}.zip"

out_dir = "./datasets"

data_path = util.download_and_unzip(url, out_dir)

corpus, queries, qrels = GenericDataLoader(data_path).load(split="test")

doc_ids = list(corpus.keys())
corpus_texts = [corpus[doc_id]["text"] for doc_id in doc_ids]

print("Corpus size:", len(corpus_texts))
print("Queries:", len(queries))

In [ ]:
model = SentenceTransformer('BAAI/bge-small-en')

print("BGE model loaded")

In [ ]:
corpus_embeddings = model.encode(
    corpus_texts,
    show_progress_bar=True,
    convert_to_numpy=True
)

print("Corpus encoded")

In [ ]:
def bge_retrieve(query, top_k=10):
    query_embedding = model.encode([query], convert_to_numpy=True)[0]

    scores = cosine_similarity([query_embedding], corpus_embeddings)[0]

    top_indices = np.argsort(scores)[::-1][:top_k]

    return [doc_ids[i] for i in top_indices]

In [ ]:
def recall_at_k(retrieved, relevant, k):
    retrieved_k = set(retrieved[:k])
    relevant_set = set(relevant)

    return len(retrieved_k & relevant_set) / len(relevant_set)

In [ ]:
recalls = []

for qid in queries:
    query = queries[qid]

    retrieved_docs = bge_retrieve(query, top_k=10)
    relevant_docs = list(qrels[qid].keys())

    if len(relevant_docs) == 0:
        continue

    recall = recall_at_k(retrieved_docs, relevant_docs, 10)
    recalls.append(recall)

avg_recall = sum(recalls) / len(recalls)

print("BGE Recall@10:", avg_recall)